In [10]:
import os
import re

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patheffects import withStroke
from matplotlib.ticker import MultipleLocator

# ============ 参数区 ============
CSV_PATH = r"D:\大学\数据分析\年度数据.csv"
OUT_PATH = "china_population.mp4"
FPS = 30
STEPS_PER_YEAR = 24
HOLD_SECONDS = 2.5
W, H, DPI = 12.8, 7.2, 100

SERIES = [
    ("年末总人口", "全国总人口", "#C8102E", 1.0, 9),
    ("城镇人口", "城镇人口", "#0E7C86", 0.9, -9),
    ("乡村人口", "乡村人口", "#2B5C8A", 0.8, -9),]

CAPTIONS = [
    (2011, 2015, "2011—2015：人口总量持续攀升，城镇化稳步推进"),
    (2016, 2017, "2016—2017：全面二孩落地，出生人口短暂回升"),
    (2018, 2020, "2018—2020：城镇化率突破 60%，乡村人口加速外流"),
    (2021, 2021, "2021：总人口 141,260 万人，达到历史峰值"),
    (2022, 2025, "2022—2025：总人口连续下降，进入负增长阶段"),]

EVENTS = {
    2011: "十二五开局",
    2012: "劳动年龄人口见顶",
    2013: "单独二孩政策",
    2014: "城镇化率 54.8%",
    2015: "全面二孩政策出台",
    2016: "全面二孩实施",
    2017: "出生人口 1723 万",
    2018: "城镇化率突破 59%",
    2019: "总人口突破 14 亿",
    2020: "第七次全国人口普查",
    2021: "总人口达到峰值",
    2022: "人口自然增长率转负",
    2023: "总人口连续两年下降",
    2024: "出生人口小幅回升",
    2025: "人口负增长深化",}


def setup_font():
    prefer = ["Microsoft YaHei", "Microsoft YaHei UI", "SimHei", "Noto Sans CJK SC", "Source Han Sans SC"]
    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in prefer:
        if name in available:
            plt.rcParams["font.sans-serif"] = [name]
            break
    plt.rcParams["axes.unicode_minus"] = False

def load_series(path):
    df = pd.read_csv(path, encoding="utf-8-sig", skiprows=2)
    df.columns = [str(c).strip() for c in df.columns]
    df = df.loc[:, [c for c in df.columns if not c.startswith("Unnamed")]]

    name_col = df.columns[0]
    df = df[df[name_col].notna()]
    df = df[df[df.columns[1:]].notna().any(axis=1)]

    df[name_col] = (df[name_col].astype(str)
                    .str.replace(r"\s*[（(].*?[)）]", "", regex=True).str.strip())
    df = df.set_index(name_col)

    years = np.array([int(re.search(r"(\d{4})", c).group(1)) for c in df.columns])
    order = np.argsort(years)
    df = df.iloc[:, order]
    years = years[order].astype(float)

    return years, {name: df.loc[name].to_numpy(dtype=float) for name in df.index}

def smooth_curve(x, y, n=600):
    try:
        from scipy.interpolate import PchipInterpolator

        xs = np.linspace(x[0], x[-1], n)
        return xs, PchipInterpolator(x, y)(xs)
    except Exception:
        xs = np.linspace(x[0], x[-1], n)
        return xs, np.interp(xs, x, y)

def render(years, data):
    y0, y1 = years[0], years[-1]
    chosen = [(k, label, color, lw, dy) for k, label, color, lw, dy in SERIES if k in data]
    if not chosen:
        raise ValueError(f"CSV 中找不到所需指标，实际可用：{list(data)}")

    curves = {key: smooth_curve(years, data[key]) for key, *_ in chosen}

    all_vals = np.concatenate([data[k] for k, *_ in chosen])
    y_lo = np.floor((all_vals.min() - 3000) / 20000) * 20000
    y_hi = all_vals.max() * 1.06

    fig = plt.figure(figsize=(W, H), dpi=DPI)
    fig.patch.set_facecolor("white")
    ax = fig.add_axes([0.075, 0.150, 0.895, 0.705])
    ax.set_facecolor("white")

    ax.set_xlim(y0 - 0.35, y1 + 1.55)
    ax.set_ylim(y_lo, y_hi)
    ax.set_xticks(years)
    ax.set_xticklabels([str(int(v)) for v in years], fontsize=9.5, color="#8C8C8C")
    ax.tick_params(axis="x", length=0, pad=6)
    ax.yaxis.set_major_locator(MultipleLocator(20000))
    ax.yaxis.set_major_formatter(lambda v, p: f"{v:,.0f}")
    ax.tick_params(axis="y", length=0, pad=10, labelsize=10.5, colors="#8C8C8C")
    ax.grid(axis="y", color="#E9E9E9", lw=1.1)
    ax.grid(axis="x", color="#F4F4F4", lw=0.7)
    ax.set_axisbelow(True)
    for s in ax.spines.values():
        s.set_visible(False)

    fig.text(0.5, 0.958, "中国人口变化 2011—2025", ha="center", va="center",
             fontsize=27, fontweight="bold", color="#1F1F1F")
    fig.text(0.072, 0.868, "单位：万人", fontsize=10, color="#B0B0B0")
    fig.text(0.075, 0.072, "数据来源：国家统计局《年度数据》", fontsize=9.5, color="#B0B0B0")

    year_text = ax.text(0.695, 0.33, "", transform=ax.transAxes, fontsize=104, color="#E2E2E2",
                        ha="center", va="center", zorder=1)
    event_text = ax.text(0.695, 0.155, "", transform=ax.transAxes, fontsize=28, color="#C6C6C6",
                         ha="center", va="center", zorder=1)
    caption_text = fig.text(0.5, 0.032, "", ha="center", va="center", fontsize=21, fontweight="bold",
                            color="#1A1A1A", zorder=20)
    caption_text.set_path_effects([withStroke(linewidth=4.5, foreground="white")])

    lines, heads, rings, labels = {}, {}, {}, {}
    for key, label, color, lw, dy in chosen:
        (lines[key],) = ax.plot([], [], color=color, lw=lw * 4.2, solid_capstyle="round",
                                solid_joinstyle="round", zorder=5, label=label)
        heads[key] = ax.scatter([], [], s=95, color=color, zorder=6, edgecolors="white", linewidths=1.6)
        rings[key] = ax.scatter([], [], s=95, color=color, alpha=0.22, zorder=4, linewidths=0)
        labels[key] = ax.text(0, 0, "", fontsize=15.5, fontweight="bold", color=color,
                              ha="left", va="center", zorder=7)

    vline = ax.axvline(y0, color="#D5D5D5", lw=1.0, ls=(0, (4, 3)), zorder=3)

    handles = [lines[k] for k, *_ in chosen]
    leg = fig.legend(handles=handles, labels=[c[1] for c in chosen], loc="upper center",
                     bbox_to_anchor=(0.5, 0.930), ncol=len(chosen), frameon=False,
                     fontsize=14, handlelength=1.6, columnspacing=2.6, handletextpad=0.6)
    for t in leg.get_texts():
        t.set_color("#3F3F3F")

    n_years = len(years)
    total_frames = int((n_years - 1) * STEPS_PER_YEAR) + int(HOLD_SECONDS * FPS)

    def frame_data(frame):
        head_x = y0 + min(frame, (n_years - 1) * STEPS_PER_YEAR) / STEPS_PER_YEAR
        pulse = 1.0 + 0.28 * np.sin(frame / FPS * 2 * np.pi * 1.1)
        return head_x, pulse

    def draw(frame):
        head_x, pulse = frame_data(frame)
        cur_year = int(round(head_x))

        for key, label, color, lw, dy in chosen:
            xs, ys = curves[key]
            head_y = float(np.interp(head_x, xs, ys))
            mask = xs <= head_x
            lines[key].set_data(xs[mask], ys[mask])
            heads[key].set_offsets([[head_x, head_y]])
            rings[key].set_offsets([[head_x, head_y]])
            rings[key].set_sizes([95 * (1.7 + 0.5 * (pulse - 1.0)) * 4])

            labels[key].set_position((head_x + 0.16, head_y + dy * 90))
            labels[key].set_text(f"{head_y:,.0f}")

        vline.set_xdata([head_x, head_x])
        year_text.set_text(str(cur_year))
        event_text.set_text(EVENTS.get(cur_year, ""))

        cap = ""
        for a, b, text in CAPTIONS:
            if a <= cur_year <= b:
                cap = text
                break
        caption_text.set_text(cap)
        return ()

    return fig, draw, total_frames

def main():
    setup_font()
    csv_path = CSV_PATH if os.path.exists(CSV_PATH) else os.path.join(
        os.path.dirname(os.path.abspath(__file__)), "年度数据.csv")
    years, data = load_series(csv_path)
    fig, draw, total_frames = render(years, data)

    try:
        import cv2

        writer = cv2.VideoWriter(OUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), FPS,
                                 (int(W * DPI), int(H * DPI)))
        if not writer.isOpened():
            raise RuntimeError("VideoWriter 打开失败")
        for i in range(total_frames):
            draw(i)
            fig.canvas.draw()
            buf = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
            writer.write(cv2.cvtColor(buf, cv2.COLOR_RGB2BGR))
            if i % 30 == 0:
                print(f"\r渲染中 {i}/{total_frames}", end="", flush=True)
        writer.release()
        target = OUT_PATH
    except Exception as exc:
        print(f"\nOpenCV 写入不可用（{exc}），改用 matplotlib 动画写入器 …")
        from matplotlib.animation import FFMpegWriter, FuncAnimation, PillowWriter

        anim = FuncAnimation(fig, draw, frames=total_frames, interval=1000 / FPS, blit=False)
        try:
            anim.save(OUT_PATH, writer=FFMpegWriter(fps=FPS, bitrate=8000))
            target = OUT_PATH
        except Exception as exc2:
            target = os.path.splitext(OUT_PATH)[0] + ".gif"
            print(f"ffmpeg 不可用（{exc2}），改导出 GIF：{target}")
            anim.save(target, writer=PillowWriter(fps=15))

    plt.close(fig)
    print(f"\n完成：{os.path.abspath(target)}")


if __name__ == "__main__":
    main()

渲染中 390/411
完成：C:\Users\zhou\数据可视化\Untitled Folder\china_population.mp4


In [11]:
import os
import re

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patheffects import withStroke
from matplotlib.ticker import MultipleLocator

# ============ 参数区 ============
CSV_PATH = r"C:\Users\zhou\数据可视化\Untitled Folder\人口出生率、死亡率和自然增长率.csv"
OUT_PATH = "birth_death_rate.mp4"
FPS = 30
STEPS_PER_YEAR = 26
HOLD_SECONDS = 2.8
W, H, DPI = 12.8, 7.2, 100

SERIES = [
    ("人口出生率", "人口出生率", "#C8102E", 1.05, +1),
    ("人口死亡率", "人口死亡率", "#0E7C86", 0.95, +1),
    ("人口自然增长率", "人口自然增长率", "#2B5C8A", 0.95, -1),]

CAPTIONS = [
    (2016, 2017, "2016—2017：出生率短暂回升，自然增长率仍在 5‰ 以上"),
    (2018, 2019, "2018—2019：出生率跌破 11‰，自然增长率向零逼近"),
    (2020, 2021, "2020—2021：出生率降至 8.52‰，自然增长率仅 0.34‰"),
    (2022, 2023, "2022—2023：出生率低于死亡率，人口自然增长转负"),
    (2024, 2025, "2024—2025：死亡率升破 8‰，负增长持续扩大"),]

EVENTS = {
    2016: "全面二孩实施",
    2017: "出生人口 1723 万",
    2018: "出生率 10.86‰",
    2019: "出生率跌破 11‰",
    2020: "第七次全国人口普查",
    2021: "自然增长率 0.34‰",
    2022: "人口自然增长转负",
    2023: "出生率 6.39‰",
    2024: "出生率小幅回升",
    2025: "死亡率 8.04‰",}

def setup_font():
    prefer = ["Microsoft YaHei", "Microsoft YaHei UI", "SimHei", "Noto Sans CJK SC", "Source Han Sans SC"]
    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in prefer:
        if name in available:
            plt.rcParams["font.sans-serif"] = [name]
            break
    plt.rcParams["axes.unicode_minus"] = False


def load_series(path):
    df = pd.read_csv(path, encoding="utf-8-sig", skiprows=2)
    df.columns = [str(c).strip() for c in df.columns]
    df = df.loc[:, [c for c in df.columns if not c.startswith("Unnamed")]]

    name_col = df.columns[0]
    df = df[df[name_col].notna()]
    df = df[df[df.columns[1:]].notna().any(axis=1)]

    df[name_col] = (df[name_col].astype(str)
                    .str.replace(r"\s*[（(].*?[)）]", "", regex=True).str.strip())
    df = df.set_index(name_col)

    years = np.array([int(re.search(r"(\d{4})", c).group(1)) for c in df.columns])
    order = np.argsort(years)
    df = df.iloc[:, order]
    years = years[order].astype(float)

    return years, {name: df.loc[name].to_numpy(dtype=float) for name in df.index}


def smooth_curve(x, y, n=600):
    try:
        from scipy.interpolate import PchipInterpolator

        xs = np.linspace(x[0], x[-1], n)
        return xs, PchipInterpolator(x, y)(xs)
    except Exception:
        xs = np.linspace(x[0], x[-1], n)
        return xs, np.interp(xs, x, y)

def render(years, data):
    y0, y1 = years[0], years[-1]
    chosen = [(k, label, color, lw, sign) for k, label, color, lw, sign in SERIES if k in data]
    if not chosen:
        raise ValueError(f"CSV 中找不到所需指标，实际可用：{list(data)}")

    curves = {key: smooth_curve(years, data[key]) for key, *_ in chosen}

    all_vals = np.concatenate([data[k] for k, *_ in chosen])
    y_lo = np.floor((all_vals.min() - 2) / 4) * 4
    y_hi = np.ceil((all_vals.max() + 2) / 4) * 4
    lab_off = 0.055 * (y_hi - y_lo)

    fig = plt.figure(figsize=(W, H), dpi=DPI)
    fig.patch.set_facecolor("white")
    ax = fig.add_axes([0.075, 0.150, 0.895, 0.705])
    ax.set_facecolor("white")

    ax.set_xlim(y0 - 0.35, y1 + 1.55)
    ax.set_ylim(y_lo, y_hi)
    ax.set_xticks(years)
    ax.set_xticklabels([str(int(v)) for v in years], fontsize=10.5, color="#8C8C8C")
    ax.tick_params(axis="x", length=0, pad=6)
    ax.yaxis.set_major_locator(MultipleLocator(4))
    ax.yaxis.set_major_formatter(lambda v, p: f"{v:,.0f}")
    ax.tick_params(axis="y", length=0, pad=10, labelsize=10.5, colors="#8C8C8C")
    ax.grid(axis="y", color="#E9E9E9", lw=1.1)
    ax.grid(axis="x", color="#F4F4F4", lw=0.7)
    ax.set_axisbelow(True)
    for s in ax.spines.values():
        s.set_visible(False)

    ax.axhline(0, color="#C4C4C4", lw=1.4, zorder=2)

    fig.text(0.5, 0.958, "中国人口出生率 · 死亡率 · 自然增长率", ha="center", va="center",
             fontsize=25, fontweight="bold", color="#1F1F1F")
    fig.text(0.075, 0.877, "2016—2025 年 · 单位：‰", ha="left", va="center",
             fontsize=10.5, color="#A8A8A8")
    fig.text(0.075, 0.072, "数据来源：国家统计局《年度数据》", fontsize=9.5, color="#B0B0B0")

    year_text = ax.text(0.63, 0.235, "", transform=ax.transAxes, fontsize=96, color="#E4E4E4",
                        ha="center", va="center", zorder=1)
    event_text = ax.text(0.63, 0.095, "", transform=ax.transAxes, fontsize=26, color="#C6C6C6",
                         ha="center", va="center", zorder=1)
    caption_text = fig.text(0.5, 0.032, "", ha="center", va="center", fontsize=21, fontweight="bold",
                            color="#1A1A1A", zorder=20)
    caption_text.set_path_effects([withStroke(linewidth=4.5, foreground="white")])

    lines, heads, rings, labels = {}, {}, {}, {}
    for key, label, color, lw, sign in chosen:
        (lines[key],) = ax.plot([], [], color=color, lw=lw * 4.2, solid_capstyle="round",
                                solid_joinstyle="round", zorder=5, label=label)
        heads[key] = ax.scatter([], [], s=95, color=color, zorder=6, edgecolors="white", linewidths=1.6)
        rings[key] = ax.scatter([], [], s=95, color=color, alpha=0.22, zorder=4, linewidths=0)
        labels[key] = ax.text(0, 0, "", fontsize=15, fontweight="bold", color=color,
                              ha="left", va="center", zorder=7)

    vline = ax.axvline(y0, color="#D5D5D5", lw=1.0, ls=(0, (4, 3)), zorder=3)

    handles = [lines[k] for k, *_ in chosen]
    leg = fig.legend(handles=handles, labels=[c[1] for c in chosen], loc="upper center",
                     bbox_to_anchor=(0.5, 0.895), ncol=len(chosen), frameon=False,
                     fontsize=13.5, handlelength=1.6, columnspacing=2.4, handletextpad=0.6)
    for t in leg.get_texts():
        t.set_color("#3F3F3F")

    n_years = len(years)
    total_frames = int((n_years - 1) * STEPS_PER_YEAR) + int(HOLD_SECONDS * FPS)

    def frame_data(frame):
        head_x = y0 + min(frame, (n_years - 1) * STEPS_PER_YEAR) / STEPS_PER_YEAR
        pulse = 1.0 + 0.28 * np.sin(frame / FPS * 2 * np.pi * 1.1)
        return head_x, pulse

    def draw(frame):
        head_x, pulse = frame_data(frame)
        cur_year = int(round(head_x))

        for key, label, color, lw, sign in chosen:
            xs, ys = curves[key]
            head_y = float(np.interp(head_x, xs, ys))
            mask = xs <= head_x
            lines[key].set_data(xs[mask], ys[mask])
            heads[key].set_offsets([[head_x, head_y]])
            rings[key].set_offsets([[head_x, head_y]])
            rings[key].set_sizes([95 * (1.7 + 0.5 * (pulse - 1.0)) * 4])

            labels[key].set_position((head_x + 0.16, head_y + sign * lab_off))
            labels[key].set_text(f"{head_y:.2f}‰")

        vline.set_xdata([head_x, head_x])
        year_text.set_text(str(cur_year))
        event_text.set_text(EVENTS.get(cur_year, ""))

        cap = ""
        for a, b, text in CAPTIONS:
            if a <= cur_year <= b:
                cap = text
                break
        caption_text.set_text(cap)
        return ()

    return fig, draw, total_frames

def main():
    setup_font()
    csv_path = CSV_PATH if os.path.exists(CSV_PATH) else os.path.join(
        os.path.dirname(os.path.abspath(__file__)), "人口出生率、死亡率和自然增长率.csv")
    years, data = load_series(csv_path)
    fig, draw, total_frames = render(years, data)

    try:
        import cv2

        writer = cv2.VideoWriter(OUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), FPS,
                                 (int(W * DPI), int(H * DPI)))
        if not writer.isOpened():
            raise RuntimeError("VideoWriter 打开失败")
        for i in range(total_frames):
            draw(i)
            fig.canvas.draw()
            buf = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
            writer.write(cv2.cvtColor(buf, cv2.COLOR_RGB2BGR))
            if i % 30 == 0:
                print(f"\r渲染中 {i}/{total_frames}", end="", flush=True)
        writer.release()
        target = OUT_PATH
    except Exception as exc:
        print(f"\nOpenCV 写入不可用（{exc}），改用 matplotlib 动画写入器 …")
        from matplotlib.animation import FFMpegWriter, FuncAnimation, PillowWriter

        anim = FuncAnimation(fig, draw, frames=total_frames, interval=1000 / FPS, blit=False)
        try:
            anim.save(OUT_PATH, writer=FFMpegWriter(fps=FPS, bitrate=8000))
            target = OUT_PATH
        except Exception as exc2:
            target = os.path.splitext(OUT_PATH)[0] + ".gif"
            print(f"ffmpeg 不可用（{exc2}），改导出 GIF：{target}")
            anim.save(target, writer=PillowWriter(fps=15))

    plt.close(fig)
    print(f"\n完成：{os.path.abspath(target)}")

if __name__ == "__main__":
    main()

渲染中 300/318
完成：C:\Users\zhou\数据可视化\Untitled Folder\birth_death_rate.mp4
